# 01. 환경 점검 및 데이터 준비 (Colab T4 / ADR-0005)

첫날 환경 점검(T4 GPU, Google Drive 마운트, 원격 Git push 테스트)과
DIV2K 패치 풀 및 테스트셋 LR 압축 파일 생성/검증을 수행하는 얇은 실행 노트북입니다.

- **주의**: 토큰이나 자격 증명은 코드/노트북에 절대 저장하지 말고 실행 시 `getpass`로만 입력합니다.

In [5]:
# 1. GPU 확인
!nvidia-smi

In [6]:
# 2. Google Drive 마운트 확인 (실패 시 ADR-0005 대체 경로: gdown 사용)
from google.colab import drive
drive.mount('/content/drive')

In [7]:
# 3. 저장소 clone 및 의존성 설치
REPO_URL = "https://github.com/minyi-k03/VideoProcessing-Project.git"
BRANCH = "main"  # 작업 시 본인 이름 브랜치로 변경

!git clone -b {BRANCH} {REPO_URL} /content/VideoProcessing-Project
%cd /content/VideoProcessing-Project
!pip install -r requirements.txt

In [9]:
# 3-1. 원격 push 점검 (T0.1, ADR-0005)
# - 토큰은 getpass로만 받는다. 파일, .git/config, 셀 출력 어디에도 남기지 않는다.
# - 점검용 임시 브랜치를 push한 뒤 바로 지운다. main에는 아무것도 올리지 않는다.
# - 실패하면 ADR-0005 대체 경로를 쓴다: 실행 기록을 내려받아 로컬 VS Code에서 커밋한다.
# - 실행 후 이 노트북의 셀 출력은 지우고 커밋한다.
import getpass, os, stat, subprocess, tempfile, time

GH_USER = input("GitHub 사용자명: ").strip()
GIT_NAME = input("커밋 작성자 이름 (이 런타임에서만 사용): ").strip()
GIT_EMAIL = input("커밋 작성자 이메일 (이 런타임에서만 사용): ").strip()
token = getpass.getpass("GitHub 토큰 (저장되지 않음): ")

# 토큰을 명령행이나 remote URL에 넣지 않고, 환경 변수를 읽는 임시 askpass 스크립트로 전달한다.
askpass = tempfile.NamedTemporaryFile("w", suffix=".sh", delete=False)
askpass.write('#!/bin/sh\ncase "$1" in\n  Username*) echo "$GH_USER";;\n  *) echo "$GH_TOKEN";;\nesac\n')
askpass.close()
os.chmod(askpass.name, stat.S_IRWXU)

env = {**os.environ, "GIT_ASKPASS": askpass.name, "GIT_TERMINAL_PROMPT": "0",
       "GH_USER": GH_USER, "GH_TOKEN": token}
ident = ["-c", f"user.name={GIT_NAME}", "-c", f"user.email={GIT_EMAIL}"]
test_branch = f"env-check/{GH_USER}-{int(time.time())}"


def git(*args, check=True):
    shown = " ".join(a for a in args if not a.startswith("user."))  # 이름·이메일은 출력하지 않는다
    r = subprocess.run(["git", *args], env=env, capture_output=True, text=True)
    print(f"$ git {shown}  -> exit {r.returncode}")
    if r.stdout.strip():
        print(r.stdout.strip())
    if r.stderr.strip():
        print(r.stderr.strip())
    if check and r.returncode != 0:
        raise RuntimeError(f"git {shown} 실패")
    return r


try:
    git("switch", "-c", test_branch)
    git(*ident, "commit", "--allow-empty", "-m", "chore: env check push test")
    git("push", "origin", test_branch)
    print("✅ push 성공")
    git("push", "origin", "--delete", test_branch)
    print("✅ 점검용 원격 브랜치 삭제 완료")
except RuntimeError as e:
    print(f"❌ {e}")
    print("   ADR-0005 대체 경로: 실행 기록을 내려받아 로컬 VS Code에서 내 Git 자격 증명으로 커밋한다.")
finally:
    git("switch", BRANCH, check=False)
    git("branch", "-D", test_branch, check=False)
    os.remove(askpass.name)
    del token, env

In [ ]:
# 4. 패치 풀 및 테스트셋 LR 생성 (1회성 실행 후 Drive/Kaggle에 압축 보관)
# !python scripts/prepare_data.py --div2k-hr-dir /content/raw/DIV2K_train_HR --testsets-hr-dir /content/raw/testsets --output-dir /content/data